In [ ]:
import anndata
import numpy as np
import scvelo as scv
import scanpy as sc
import torch
import latentvelo as ltv
import pickle as pickle
import matplotlib.pyplot as plt
import pandas as pd

## Gastrulation erythroid

In [ ]:
adata = sc.read_h5ad('../Data/gastrulation_erythroid.h5ad')
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=3000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

adata.obs_names = adata.obs_names.astype(str)
adata = ltv.utils.standard_clean_recipe(adata, spliced_key='spliced', unspliced_key='unspliced', normalize_library=True, n_top_genes=2000, log=False)

model = ltv.models.VAE(observed=2000, latent_dim=20, zr_dim=3, h_dim=4, encoder_hidden=25)
epochs, val_ae, val_traj = ltv.train(model, adata, batch_size=250, epochs=50, name='Gastrulation_Erythroid', learning_rate=1e-2, grad_clip=100)

latent_adata, adata = ltv.output_results(model, adata, gene_velocity=True, decoded=True, embedding='umap')
latent_adata.obsm['X_umap'] = adata.obsm['X_umap']
scv.tl.velocity_graph(latent_adata, vkey='spliced_velocity', n_jobs=-1)
scv.tl.velocity_embedding(latent_adata, basis='umap', vkey='spliced_velocity')
scv.tl.velocity_graph(adata, vkey='velo', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='velo')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
#scv.pl.velocity_embedding_stream(latent_adata, basis='umap', save = False, vkey='spliced_velocity', color='celltype', title='LatentVelo', fontsize=20, arrow_size=1.8, show = False, ax = ax, legend_fontsize = 15)
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velo', color='celltype', title='LatentVelo', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Gastrulation_Erythroid_Results/LatentVelo_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Gastrulation_Erythroid_Results/LatentVelo.h5ad')

## Dentate Gyrus neurogenesis

In [ ]:
adata = sc.read_h5ad('../Data/DentateGyrus_10X43_1.h5ad')
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=3000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

adata.obs_names = adata.obs_names.astype(str)
adata = ltv.utils.standard_clean_recipe(adata, spliced_key='spliced', unspliced_key='unspliced', normalize_library=True, n_top_genes=2000, log=True)

model = ltv.models.VAE(observed=2000, latent_dim=20, zr_dim=3, h_dim=4, encoder_hidden=25)
epochs, val_ae, val_traj = ltv.train(model, adata, batch_size=250, epochs=50, name='Dentate_Gyrus', learning_rate=1e-2, grad_clip=100)

latent_adata, adata = ltv.output_results(model, adata, gene_velocity=True, decoded=True, embedding='umap')
latent_adata.obsm['X_umap'] = adata.obsm['X_umap']
scv.tl.velocity_graph(latent_adata, vkey='spliced_velocity', n_jobs=-1)
scv.tl.velocity_embedding(latent_adata, basis='umap', vkey='spliced_velocity')
scv.tl.velocity_graph(adata, vkey='velo', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='velo')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
#scv.pl.velocity_embedding_stream(latent_adata, basis='umap', save = False, vkey='spliced_velocity', color='clusters', title='LatentVelo', fontsize=20, arrow_size=1.8, show = False, ax = ax, legend_fontsize = 15)
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velo', color='clusters', title='LatentVelo', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Dentate_Gyrus_Results/LatentVelo_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Dentate_Gyrus_Results/LatentVelo.h5ad')

## Simulated data

In [ ]:
adata = sc.read_h5ad('../Data/simulated_T_cell.h5ad')
adata.obsm['X_umap'] = pickle.load(open('../Data/UMAP.pkl','rb'))
scv.pp.moments(adata, n_neighbors=30)

adata.obs_names = adata.obs_names.astype(str)
adata = ltv.utils.standard_clean_recipe(adata, spliced_key='spliced', unspliced_key='unspliced', normalize_library=True, n_top_genes=3, log=False, smooth=True, umap=False, celltype_key=None, r2_adjust=True)

model = ltv.models.VAE(observed = 3, latent_dim = 20, zr_dim = 4, h_dim = 4)
epochs, val_ae, val_traj = ltv.train(model, adata, batch_size=250, epochs=50, name='TME', grad_clip=100)

latent_adata, adata = ltv.output_results(model, adata, gene_velocity=True, decoded=True, embedding='umap')
latent_adata.obsm['X_umap'] = pickle.load(open('../Data/UMAP.pkl','rb'))
scv.tl.velocity_graph(latent_adata, vkey='spliced_velocity', n_jobs=-1)
scv.tl.velocity_embedding(latent_adata, basis='umap', vkey='spliced_velocity')
scv.tl.velocity_graph(adata, vkey='velo', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='velo')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
#scv.pl.velocity_embedding_stream(latent_adata, basis='umap', save = False, vkey='spliced_velocity', color='time', title='LatentVelo', fontsize=20, arrow_size=1.8, show = False, ax = ax, legend_fontsize = 15)
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velo', color='time', title='LatentVelo', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Simulation_Results/LatentVelo_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Simulation_Results/LatentVelo.h5ad')